# PLN Projeto 1

In [199]:
import pandas as pd
import re
import uuid

df_cases = pd.read_csv("../../data/external/cases.csv")
df_meta = pd.read_csv("../../data/external/metadata.csv")
# Cruzamento dos dados: Adiciona metadados aos casos
df_merged = df_cases.merge(df_meta, on='article_id', how='left')

In [200]:
token_re = re.compile(r"""
    \d+(?:\.\d+)?(?:\s*x\s*10\d*)?
    |[a-zA-ZÀ-ÿ]+
    |[.,;:!?()%/]
""", re.VERBOSE)

def tokenize(sent):
    return [(m.group(), m.start(), m.end()) for m in token_re.finditer(sent)]

In [201]:
def normalize(tok):
    return tok.lower().rstrip('sáéíóú')  # naive, mas cobre plural/acento simples

In [ ]:
def build_gazetteers():
    return {
        'Diagnosis': ['diabetes mellitus', 'hypertension', 'acute pancreatitis', 'chronic pancreatitis', 'pancreatic pseudocyst', 'pseudocyst', 'solid mass', 'type 2 diabetes', 'tumor', 'cancer',
            'acute respiratory distress syndrome', 'sepsis', 'septic shock', 'pneumonia', 'pulmonary embolism', 'acute heart failure', 'congestive heart failure', 'myocardial infarction',
            'ischemic stroke', 'hemorrhagic stroke', 'acute kidney injury', 'chronic kidney disease', 'renal failure', 'acute liver failure', 'cirrhosis', 'hepatitis',
            'cholangitis', 'biliary obstruction', 'appendicitis', 'peritonitis', 'gastrointestinal bleeding', 'intestinal obstruction', 'peptic ulcer disease',
            'deep vein thrombosis', 'venous thromboembolism', 'anemia', 'thrombocytopenia', 'leukemia', 'lymphoma', 'hemolytic anemia', 'fatty liver disease',
            'metastatic disease', 'meningitis', 'encephalitis', 'pyelonephritis', 'urinary tract infection', 'bronchitis', 'asthma', 'pneumonitis', 'respiratory failure',
            'arrhythmia', 'atrial fibrillation', 'valvular disease', 'endocarditis', 'pericarditis', 'liver abscess', 'splenomegaly'
        ],
        'Symptom': ['epigastric pain', 'nausea', 'fever', 'epigastric tenderness', 'fat stranding', 'pain', 'headache', 'cough',
            'abdominal pain', 'vomiting', 'diarrhea', 'constipation', 'bloody stools', 'hematochezia', 'melena', 'shortness of breath', 'dyspnea', 'wheezing',
            'chest pain', 'palpitations', 'dizziness', 'syncope', 'fatigue', 'weakness', 'malaise', 'loss of appetite', 'anorexia', 'weight loss', 'weight gain',
            'swelling', 'edema', 'leg swelling', 'lower extremity edema', 'jaundice', 'dark urine', 'itching', 'pruritus', 'confusion', 'delirium', 'sleep disturbance',
            'insomnia', 'restlessness', 'anxiety', 'depression', 'tremor', 'muscle weakness', 'myalgia', 'arthralgia', 'back pain', 'joint pain', 'neck pain',
            'sore throat', 'hoarseness', 'hemoptysis', 'cyanosis', 'hematuria', 'dysuria', 'polyuria', 'oliguria', 'urinary retention', 'flank pain', 'cold intolerance',
            'heat intolerance', 'blurred vision', 'visual disturbance', 'vertigo', 'paresthesia', 'numbness', 'tingling', 'bruising', 'easy bruising', 'bleeding', 'nosebleed',
            'productive cough', 'nonproductive cough', 'dysphagia', 'odynophagia', 'hiccups', 'tachycardia', 'hypotension', 'hypertension symptoms', 'chills', 'rigors', 'sweating',
            'night sweats', 'burning sensation', 'pressure sensation', 'abdominal distension', 'bloating', 'gastroesophageal reflux', 'heartburn', 'dyspepsia', 'gastrointestinal pain', 'thirst',
            'polyuria', 'excessive thirst', 'hyperglycemia symptoms', 'hypoglycemia symptoms', 'fainting'
        ],
        'Exam': [
          'computed tomography', 'ct', 'thoracic ct', 'endoscopic ultrasound', 'eus', 'ultrasound', 'laboratory tests', 'mri', 'x-ray', 'echocardiogram', 'echocardiography', 'ecg', 'flow cytometry',
          'hemoglobin', 'hematocrit', 'white blood cell count', 'total leukocyte count', 'leucocytosis', 'lymphocyte count', 'platelet count', 'platelet counts', 'mean corpuscular volume', 
          'reticulocyte count', 'haptoglobin', 'adamts13 activity', 'c-reactive protein', 'crp', 'erythrocyte sedimentation rate', 'esr', 'creatinine', 'serum creatinine', 'albumin', 'total protein',
          'ferritin', 'lactate dehydrogenase', 'ldh', 'complement c3', 'complement c4', 'c3', 'c4', 'troponin', 'troponin i', 'ck-mb', 'creatine kinase', 'myoglobin', 'ejection fraction', 'lvef', 
          'mean gradient', 'nt-probnp', 'heart rate', 'blood pressure', 'mean arterial blood pressure', 'glycosylated hemoglobin', 'hemoglobin a1c', 'hba1c', 'blood sugar', 'immunoglobulin m', 
          'immunoglobulin g', 'igm', 'igg', 'igg levels', 'forced vital capacity', 'fvc', 'forced expiratory volume', 'fev1', 'diffusing capacity', 'dlco', 'oxygen saturation', 'respiratory rate', 
          'carcinoembryonic antigen', 'cea', 'pd-l1 tps score','hydroxychloroquine level', 'gtt',
      ],
        'Treatment': [
            'intravenous fluids', 'oral rehydration', 'analgesia', 'analgesics', 'pain control',
            'antibiotics', 'antiviral therapy', 'antifungal therapy', 'antiparasitic therapy',
            'anticoagulation', 'antiplatelet therapy', 'heparin', 'warfarin', 'enoxaparin',
            'insulin', 'metformin', 'glucose control', 'blood pressure control', 'antihypertensive therapy',
            'fluid resuscitation', 'electrolyte replacement', 'oxygen therapy', 'ventilation support',
            'corticosteroids', 'prednisone', 'hydrocortisone', 'dexamethasone', 'steroid therapy',
            'immunosuppressive therapy', 'plasma exchange', 'plasmapheresis', 'intravenous immunoglobulin', 'ivig',
            'chemotherapy', 'radiotherapy', 'targeted therapy', 'immunotherapy', 'hormonal therapy',
            'surgery', 'laparotomy', 'laparoscopy', 'resection', 'drainage', 'debridement',
            'endoscopic drainage', 'cystgastrostomy', 'gastrojejunostomy', 'stent placement', 'biliary drainage',
            'dialysis', 'renal replacement therapy', 'liver transplant', 'kidney transplant', 'transplantation',
            'blood transfusion', 'platelet transfusion', 'erythrocyte transfusion',
            'physical therapy', 'rehabilitation', 'respiratory physiotherapy', 'occupational therapy',
            'bronchodilator therapy', 'inhaled beta agonist', 'nebulized therapy', 'smoking cessation',
            'prophylactic antibiotics', 'empiric therapy', 'supportive care', 'palliative care',
            'nutritional support', 'enteral nutrition', 'parenteral nutrition', 'oral nutrition',
            'thrombolysis', 'vasopressor support', 'inotropic support', 'antiemetics', 'anti-inflammatory therapy',
            'antihistamines', 'acid suppression', 'proton pump inhibitor', 'ppi', 'antacids',
            'beta-blocker therapy', 'statin therapy', 'aspirin', 'ibuprofen', 'acetaminophen', 'paracetamol',
            'antidepressant therapy', 'antiepileptic therapy', 'antiarrhythmic therapy', 'diuretic therapy',
            'vaccination', 'immunization', 'monitoring', 'observation', 'watchful waiting', 'conservative management'
        ]
    }

In [203]:
def build_lookup(gazetteer):
    lookup = {}
    for ent_type, keywords in gazetteer.items():
        for kw in keywords:
            for tok in tokenize(kw):
                lookup[normalize(tok[0])] = (ent_type, kw)
    print(lookup)
    return lookup

In [204]:
gazetteer = build_gazetteers()
lookup = build_lookup(gazetteer)

{'diabete': ('Diagnosis', 'type 2 diabetes'), 'mellitu': ('Diagnosis', 'diabetes mellitus'), 'hypertension': ('Diagnosis', 'hypertension'), 'acute': ('Diagnosis', 'acute pancreatitis'), 'pancreatiti': ('Diagnosis', 'chronic pancreatitis'), 'chronic': ('Diagnosis', 'chronic pancreatitis'), 'pancreatic': ('Diagnosis', 'pancreatic pseudocyst'), 'pseudocyst': ('Diagnosis', 'pseudocyst'), 'solid': ('Diagnosis', 'solid mass'), 'ma': ('Diagnosis', 'solid mass'), 'type': ('Diagnosis', 'type 2 diabetes'), '2': ('Diagnosis', 'type 2 diabetes'), 'tumor': ('Diagnosis', 'tumor'), 'cancer': ('Diagnosis', 'cancer'), 'epigastric': ('Symptom', 'epigastric tenderness'), 'pain': ('Symptom', 'pain'), 'nausea': ('Symptom', 'nausea'), 'fever': ('Symptom', 'fever'), 'tenderne': ('Symptom', 'epigastric tenderness'), 'fat': ('Symptom', 'fat stranding'), 'stranding': ('Symptom', 'fat stranding'), 'headache': ('Symptom', 'headache'), 'cough': ('Symptom', 'cough'), 'computed': ('Exam', 'computed tomography'), 'to

In [205]:
def create_node(all_nodes, node_id, node_type, node_label, **attributes):
    node = {
        "node_id": node_id,
        "type": node_type,
        "label": node_label,
        "attributes": attributes,
    }
    all_nodes.append(node)
    return node

In [206]:
def create_edge(all_edges, edge_id, source_id, target_id, edge_type, relation, **attributes):
    edge = {
        "edge_id": edge_id,
        "source": source_id,
        "target": target_id,
        "type": edge_type,
        "relation": relation,
        "attributes": attributes,
    }
    all_edges.append(edge)
    return edge

In [207]:
def process_multicare_dataset(row, lookup):
    all_nodes = []
    all_edges = []
    seen_edges = set()

    case_id = row["case_id"]
    text = row["case_text"]

    # Creates patient node
    patient_id = f"P_{case_id}"
    age = row.get('age', 'Unknown')
    gender = row.get('gender', 'Unknown')
    create_node(all_nodes, patient_id, 'Patient', f'Case {case_id}', age=age, gender=gender)

    entity_map = {}
    valid_units = {'u/l', 'mg/l', 'ng/ml', 'mmol/l', 'g/dl', '%', 'au/ml', 'mmhg', 'l'}
    # Regex for numbers and scientific notation
    num_re = re.compile(r'\d+(?:\.\d+)?(?:\s*x\s*10\d*)?')

    sentences = re.split(r'(?<=[.!?])\s+', text)

    for sent in sentences:
        tokens = tokenize(sent)
        last_node_id = patient_id
        last_type = None

        for i, (tok, start, end) in enumerate(tokens):
            key = normalize(tok)
            if key not in lookup:
                continue
            ent_type, keyword = lookup[key]

            if keyword not in entity_map:
                node_id = f"{ent_type[:3].upper()}_{uuid.uuid4().hex[:6]}"
                create_node(all_nodes, node_id, ent_type, keyword.capitalize())
                entity_map[keyword] = node_id

            node_id = entity_map[keyword]

            relation = 'ASSOCIATED_WITH'
            if ent_type == 'Diagnosis': relation = 'DIAGNOSED_WITH'
            elif ent_type == 'Symptom': relation = 'HAS_SYMPTOM'
            elif ent_type == 'Exam': relation = 'UNDERWENT_EXAM'
            elif ent_type == 'Treatment': relation = 'TREATED_BY'

            edge_key = (patient_id, node_id, relation)
            if edge_key not in seen_edges:
                seen_edges.add(edge_key)
                create_edge(all_edges, f"e_{uuid.uuid4().hex[:6]}",
                            patient_id, node_id, ent_type, relation, source='regex_dict')

            if ent_type == 'Exam':
                last_node_id, last_type = node_id, 'Exam'

                for j in range(i + 1, len(tokens)):
                    next_tok = tokens[j][0]
                    if normalize(next_tok) in lookup:
                        break
                    if num_re.fullmatch(next_tok) and j + 1 < len(tokens):
                        unit_tok = tokens[j + 1][0].lower().lstrip('/')
                        if unit_tok in valid_units:
                            res_id = f"VAL_{uuid.uuid4().hex[:6]}"
                            create_node(all_nodes, res_id, 'ExamResult',
                                        f'{next_tok} {unit_tok}', value=next_tok, unit=unit_tok)

                            exam_edge_key = (last_node_id, res_id, 'HAS_RESULT')
                            if exam_edge_key not in seen_edges:
                                seen_edges.add(exam_edge_key)
                                create_edge(all_edges, f"e_{uuid.uuid4().hex[:6]}",
                                            last_node_id, res_id, 'ExamResult',
                                            'HAS_RESULT', source='regex_extraction')
                            break

    return pd.DataFrame(all_nodes), pd.DataFrame(all_edges)

In [208]:
df_nodes, df_edges = process_multicare_dataset(df_merged.iloc[6], lookup)  # Processa apenas o caso de índice 10 como exemplo)
df_nodes.to_csv("../../data/processed/nodes.csv", index=False)
df_edges.to_csv("../../data/processed/edges.csv", index=False)
print(f"Extração em lote concluída! {len(df_nodes)} nós e {len(df_edges)} arestas criadas.")

Extração em lote concluída! 32 nós e 31 arestas criadas.


In [209]:
def to_mermaid(nodes, edges) -> str:
    lines = ["flowchart LR"]
    for n in nodes.itertuples():
        label = str(n.label).replace('"', "'")
        lines.append(f'  {n.node_id}["{n.type}<br/>{label}"]')
    for e in edges.itertuples():
        lines.append(f'  {e.source} -->|{e.relation}| {e.target}')
    return "\n".join(lines)

In [210]:
from IPython.display import display, Markdown

mermaid_md = to_mermaid(df_nodes, df_edges)
display(Markdown(f"```mermaid\n{mermaid_md}\n```"))

```mermaid
flowchart LR
  P_PMC4835621_01["Patient<br/>Case PMC4835621_01"]
  EXA_24f430["Exam<br/>Hemoglobin a1c"]
  SYM_50048e["Symptom<br/>Headache"]
  SYM_194a59["Symptom<br/>Fever"]
  SYM_069569["Symptom<br/>Nausea"]
  DIA_29098c["Diagnosis<br/>Hypertension"]
  EXA_77c91c["Exam<br/>Laboratory tests"]
  EXA_54dc6d["Exam<br/>Forced vital capacity"]
  EXA_12405f["Exam<br/>Heart rate"]
  EXA_bbe19c["Exam<br/>Thoracic ct"]
  EXA_bfeacf["Exam<br/>White blood cell count"]
  EXA_3b2c20["Exam<br/>Blood sugar"]
  EXA_f15005["Exam<br/>Reticulocyte count"]
  VAL_68c02a["ExamResult<br/>73 %"]
  EXA_66614b["Exam<br/>Immunoglobulin g"]
  EXA_a1b34e["Exam<br/>Mean arterial blood pressure"]
  EXA_8c6724["Exam<br/>Mean corpuscular volume"]
  EXA_22efdb["Exam<br/>Forced expiratory volume"]
  VAL_a09b83["ExamResult<br/>2.4 %"]
  EXA_0b6892["Exam<br/>Hydroxychloroquine level"]
  EXA_e678de["Exam<br/>Serum creatinine"]
  EXA_7ba92a["Exam<br/>Lactate dehydrogenase"]
  EXA_9b4964["Exam<br/>Ldh"]
  EXA_e5cc22["Exam<br/>Pd-l1 tps score"]
  EXA_7a4f81["Exam<br/>Haptoglobin"]
  EXA_61b2ee["Exam<br/>Adamts13 activity"]
  VAL_b77098["ExamResult<br/>10 %"]
  EXA_acf1f5["Exam<br/>Computed tomography"]
  EXA_0a5f3a["Exam<br/>Platelet counts"]
  DIA_4aed6a["Diagnosis<br/>Chronic pancreatitis"]
  DIA_2d5fa4["Diagnosis<br/>Type 2 diabetes"]
  VAL_9c06aa["ExamResult<br/>2.5 %"]
  P_PMC4835621_01 -->|UNDERWENT_EXAM| EXA_24f430
  P_PMC4835621_01 -->|HAS_SYMPTOM| SYM_50048e
  P_PMC4835621_01 -->|HAS_SYMPTOM| SYM_194a59
  P_PMC4835621_01 -->|HAS_SYMPTOM| SYM_069569
  P_PMC4835621_01 -->|DIAGNOSED_WITH| DIA_29098c
  P_PMC4835621_01 -->|UNDERWENT_EXAM| EXA_77c91c
  P_PMC4835621_01 -->|UNDERWENT_EXAM| EXA_54dc6d
  P_PMC4835621_01 -->|UNDERWENT_EXAM| EXA_12405f
  P_PMC4835621_01 -->|UNDERWENT_EXAM| EXA_bbe19c
  P_PMC4835621_01 -->|UNDERWENT_EXAM| EXA_bfeacf
  P_PMC4835621_01 -->|UNDERWENT_EXAM| EXA_3b2c20
  P_PMC4835621_01 -->|UNDERWENT_EXAM| EXA_f15005
  EXA_f15005 -->|HAS_RESULT| VAL_68c02a
  P_PMC4835621_01 -->|UNDERWENT_EXAM| EXA_66614b
  P_PMC4835621_01 -->|UNDERWENT_EXAM| EXA_a1b34e
  P_PMC4835621_01 -->|UNDERWENT_EXAM| EXA_8c6724
  P_PMC4835621_01 -->|UNDERWENT_EXAM| EXA_22efdb
  EXA_f15005 -->|HAS_RESULT| VAL_a09b83
  P_PMC4835621_01 -->|UNDERWENT_EXAM| EXA_0b6892
  P_PMC4835621_01 -->|UNDERWENT_EXAM| EXA_e678de
  P_PMC4835621_01 -->|UNDERWENT_EXAM| EXA_7ba92a
  P_PMC4835621_01 -->|UNDERWENT_EXAM| EXA_9b4964
  P_PMC4835621_01 -->|UNDERWENT_EXAM| EXA_e5cc22
  P_PMC4835621_01 -->|UNDERWENT_EXAM| EXA_7a4f81
  P_PMC4835621_01 -->|UNDERWENT_EXAM| EXA_61b2ee
  EXA_77c91c -->|HAS_RESULT| VAL_b77098
  P_PMC4835621_01 -->|UNDERWENT_EXAM| EXA_acf1f5
  P_PMC4835621_01 -->|UNDERWENT_EXAM| EXA_0a5f3a
  P_PMC4835621_01 -->|DIAGNOSED_WITH| DIA_4aed6a
  P_PMC4835621_01 -->|DIAGNOSED_WITH| DIA_2d5fa4
  EXA_0b6892 -->|HAS_RESULT| VAL_9c06aa
```